## Import the data

Run the following cell to import all the necessary classes, functions, and packages you need for this lab.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

Import the `'data/pipelines_lab/winequality-red.csv'` dataset and print the first five rows of the data.

In [ ]:
# Import the data
df = None


# Print the first five rows


Use the `.describe()` method to print the summary stats of all columns in `df`. Pay close attention to the range (min and max values) of all columns. What do you notice?

In [ ]:
# Print the summary stats of all columns


As you can see from the data, not all features are on the same scale. Since we will be using k-nearest neighbors, which uses the distance between features to classify points, we need to bring all these features to the same scale. This can be done using standardization. 



However, before standardizing the data, let's split it into training and test sets. 

> Note: You should always split the data before applying any scaling/preprocessing techniques in order to avoid data leakage. If you don't recall why this is necessary, you should refer to the **KNN with scikit-learn - Lab.** 

## Split the data 

- Assign the target (`'quality'` column) to `y` 
- Drop this column and assign all the predictors to `X` 
- Split `X` and `y` into 75/25 training and test sets. Set `random_state` to 42

In [ ]:
# Split the predictor and target variables
y = None
X = None

# Split into training and test sets
X_train, X_test, y_train, y_test = None

## Standardize your data 

- Instantiate a `StandardScaler()` 
- Transform and fit the training data 
- Transform the test data

In [ ]:
# Instantiate StandardScaler
scaler = None

# Transform the training and test sets
scaled_data_train = None
scaled_data_test = None

# Convert into a DataFrame
scaled_df_train = pd.DataFrame(scaled_data_train, columns=X_train.columns)
scaled_df_train.head()

## Train a model 

- Instantiate a `KNeighborsClassifier()` 
- Fit the classifier to the scaled training data

In [ ]:
# Instantiate KNeighborsClassifier
clf = None

# Fit the classifier


Use the classifier's `.score()` method to calculate the accuracy on the test set (use the scaled test data)

In [ ]:
# Print the accuracy on test set


Nicely done. This pattern (preprocessing and fitting models) is very common. Although this process is fairly straightforward once you get the hang of it, **pipelines** make this process simpler, intuitive, and less error-prone. 

Instead of standardizing and fitting the model separately, you can do this in one step using `sklearn`'s `Pipeline()`. A pipeline takes in any number of preprocessing steps, each with `.fit()` and `transform()` methods (like `StandardScaler()` above), and a final step with a `.fit()` method (an estimator like `KNeighborsClassifier()`). The pipeline then sequentially applies the preprocessing steps and finally fits the model. Do this now.   

## Build a pipeline (I) 

Build a pipeline with two steps: 

- First step: `StandardScaler()` 
- Second step (estimator): `KNeighborsClassifier()`

In [ ]:
# Build a pipeline with StandardScaler and KNeighborsClassifier
scaled_pipeline_1 = None

- Transform and fit the model using this pipeline to the training data (you should use `X_train` here) 
- Print the accuracy of the model on the test set (you should use `X_test` here)

In [ ]:
# Fit the training data to pipeline


# Print the accuracy on test set


If you did everything right, this answer should match the one from above! 

Of course, you can also perform a grid search to determine which combination of hyperparameters can be used to build the best possible model. The way you define the pipeline still remains the same. What you need to do next is define the grid and then use `GridSearchCV()`. Let's do this now.

## Build a pipeline (II)

Again, build a pipeline with two steps: 

- First step: `StandardScaler()` 
- Second step (estimator): `RandomForestClassifier()`. Set `random_state=123` when instantiating the random forest classifier

In [ ]:
# Build a pipeline with StandardScaler and RandomForestClassifier
scaled_pipeline_2 = None

Use the defined `grid` to perform a grid search. We limited the hyperparameters and possible values to only a few values in order to limit the runtime.

In [ ]:
# Define the grid
grid = [{'RF__max_depth': [4, 5, 6], 
         'RF__min_samples_split': [2, 5, 10], 
         'RF__min_samples_leaf': [1, 3, 5]}]

Define a grid search now. Use: 
- the pipeline you defined above (`scaled_pipeline_2`) as the estimator 
- the parameter `grid` 
- `'accuracy'` to evaluate the score 
- 5-fold cross-validation

In [ ]:
# Define a grid search
gridsearch = None

After defining the grid values and the grid search criteria, all that is left to do is fit the model to training data and then score the test set. Do this below:

In [ ]:
# Fit the training data


# Print the accuracy on test set


## Why SHAP?

Tree-based `.feature_importances_` tells you which features were used most across splits — but it doesn't tell you *how* each feature affected any individual prediction, and it can be biased toward high-cardinality features.

**SHAP (SHapley Additive exPlanations)** assigns each feature a contribution value for each prediction, grounded in cooperative game theory. Key properties:
- **Consistent:** if a feature has more impact in model B than A, its SHAP value will always be higher in B
- **Additive:** SHAP values sum to the prediction (relative to the base rate)
- **Local + global:** explain a single prediction *or* aggregate across the dataset

## Installation

```bash
pip install shap
```

## TreeExplainer — fast path for tree models

`shap.TreeExplainer` works natively with sklearn trees, Random Forests, GradientBoostingClassifier, XGBoost, LightGBM, and CatBoost.

In [ ]:
import shap
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

## Global feature importance — summary plot

In [ ]:
shap.summary_plot(shap_values, X_test)

Each dot is one sample. Colour = feature value (red = high, blue = low). X-axis = SHAP value (positive = pushes toward class 1).

## Local explanation — single prediction

In [ ]:
# Explain the first test sample
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_test.iloc[0],
    matplotlib=True
)

## Bar plot — mean absolute SHAP (global ranking)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type='bar')

## Using SHAP with a sklearn Pipeline

When a pipeline includes preprocessing steps, pass the raw (pre-transformed) data to a `KernelExplainer`, or extract the fitted estimator and pass transformed data to `TreeExplainer`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42)),
])
pipe.fit(X_train, y_train)

# Transform data then explain the fitted estimator directly
X_test_transformed = pipe[:-1].transform(X_test)
explainer_pipe = shap.TreeExplainer(pipe['clf'])
shap_values_pipe = explainer_pipe.shap_values(X_test_transformed)

shap.summary_plot(shap_values_pipe, X_test_transformed, feature_names=X_test.columns.tolist())

## KernelExplainer — model-agnostic (slower)

Use `KernelExplainer` for models that `TreeExplainer` doesn't support (SVMs, logistic regression, etc.). It's slower — use a background sample.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)

background = shap.sample(X_train, 100)  # summarise background with 100 samples
kernel_explainer = shap.KernelExplainer(lr.predict_proba, background)
shap_values_lr = kernel_explainer.shap_values(X_test.iloc[:50])  # subset for speed

shap.summary_plot(shap_values_lr[1], X_test.iloc[:50])  # class 1 SHAP values

## Quick reference

| Model type | Recommended explainer | Speed |
|---|---|---|
| sklearn trees, RF, GBM | `TreeExplainer` | Fast |
| XGBoost, LightGBM, CatBoost | `TreeExplainer` | Fast |
| Linear models | `LinearExplainer` | Fast |
| SVM, any black-box | `KernelExplainer` | Slow |

**Further reading:** [shap.readthedocs.io](https://shap.readthedocs.io)